# 01_setup：交易設定 v1 與驗證協定

設定與協定見 `setup.yaml`；程式在 `src/s01_setup.py`。預設 `RUN = False` 只讀已存的結果。

In [1]:
import os, sys, json
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT); sys.path.insert(0, str(ROOT))
import pandas as pd
from IPython.display import Image, Markdown, display
pd.set_option("display.max_columns", 40); pd.set_option("display.width", 200)
R = ROOT / "results"
def show(name, n=60):
    p = R / f"{name}.csv"
    display(Markdown(f"**{name}**")); display(pd.read_csv(p, encoding="utf-8-sig").head(n)) if p.exists() else print("缺", name)
def fig(name, w=800):
    p = R / "figures" / name
    display(Image(filename=str(p), width=w)) if p.exists() else print("缺圖", name)

In [2]:
RUN = False
if RUN:
    from src import s01_setup
    s01_setup.main()

In [3]:
print((ROOT / "setup.yaml").read_text(encoding="utf-8")[:4000])
show("01_protocol"); show("01_splits")

# 交易設定 v1（ML4T 期中）——在下載任何資料、執行任何回測之前寫定並提交 git。
# 參數調整留在 v1；機制改變（例如只做多 → 多空、加入新的成本項）就升版 v2，結果不可直接比較。
setup_version: v1
strategy: tw-peer-diffusion
created: "2026-11-07"
author: ntutidslab

hypothesis:
  mechanism: >-
    經濟上相關的股票（同一供應鏈、同一需求來源）會受到共同的消息衝擊，但投資人注意力有限、
    小型與冷門股的消息消化較慢，所以同族群股票近期的報酬會領先預告落後者未來的報酬。
    個股本身的價格路徑（中期動能、接近前高、波動）反映投資人對新資訊的反應不足與定錨，與族群效果互補。
  source_of_edge: >-
    資訊擴散（informational / flow predictability）：對手是注意力有限、反應慢的投資人；
    台股散戶比重高，同族群消息常先反映在大型或受關注的股票上。
  literature: [Moskowitz-Grinblatt 1999 產業動能, Hou 2007 產業內領先落後, Cohen-Frazzini 2008 經濟連結, George-Hwang 2004 52 週高點]
  falsification:
    - 驗證期所有模型的每日 Spearman IC 的 Newey-West 95% CI 都包含 0 → 停止，不跑 holdout
    - 族群特徵在 triage 全部 STOP（BH 校正後無一顯著）→ 族群擴散機制不成立（即使模型有效，也照實寫）
    - 族群特徵的 IC 方向與假說相反（同伴過去漲 → 本股未來跌）→ 機制被否定
  prior_exposure: >-
    研究者在本期中之前做過同主題的探索（不同程式碼、不同設定），看過 2010–2026 台股整體走勢與部分特徵的表現。
    無法消除這個事前知識；本次的處理方式是：所有設定在看資料前寫入本檔並提交 git，holdout 的資料由程式隔離，
    只在最後執行一次；事前探索的大約試驗次數另列，用於 Deflated Sharpe 的敏感度分析。

univ

**01_protocol**

,項目,承諾
0,訓練視窗,擴張視窗，自 2007-01-02 起
1,步長／重訓頻率,126 個交易日；每個 fold 重訓一次
2,標籤期間,20 個交易日（t+1 開盤 → t+21 開盤）
3,最長特徵回看,250 個交易日（near_high250）
4,Purge／Embargo,21 個交易日／0（訓練永遠早於測試）
5,超參數,巢狀：每個 fold 訓練資料最後 126 日（再 purge 21 日）當內部驗證
6,驗證（樣本外 walk-forward）,2012-01-02 ~ 2024-10-31
7,Holdout（封存，只跑一次）,2024-11-01 ~ 2026-10-30（最後 24 個月）
8,確認性檢定,部署模型在 holdout 的每日 Spearman IC，NW 95% CI 下界 > 0


**01_splits**

,期間,fold,測試起,測試迄,訓練迄（purge 後）,內部驗證
0,驗證,1,2012-01-02,2012-07-10,2011-12-01,2011-06-07 ~ 2011-12-01
1,驗證,2,2012-07-11,2013-01-08,2012-06-11,2011-12-02 ~ 2012-06-11
2,驗證,3,2013-01-09,2013-07-19,2012-12-06,2012-06-12 ~ 2012-12-06
3,驗證,4,2013-07-22,2014-01-20,2013-06-20,2012-12-07 ~ 2013-06-20
4,驗證,5,2014-01-21,2014-07-30,2013-12-19,2013-06-21 ~ 2013-12-19
5,驗證,6,2014-07-31,2015-01-28,2014-06-30,2013-12-20 ~ 2014-06-30
6,驗證,7,2015-01-29,2015-08-10,2014-12-26,2014-07-01 ~ 2014-12-26
7,驗證,8,2015-08-11,2016-02-17,2015-07-09,2014-12-29 ~ 2015-07-09
8,驗證,9,2016-02-18,2016-08-22,2016-01-08,2015-07-13 ~ 2016-01-08
9,驗證,10,2016-08-23,2017-03-06,2016-07-22,2016-01-11 ~ 2016-07-22
